# Human lung cancer train

Set paths and samples in `config.yaml`, then run the cells in order.


## Settings


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "experiments").is_dir() and (path / "lrpairs").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

from _shared.runtime import training_epochs, training_routes, scanvi_train_kwargs
from pathlib import Path
import yaml

DATASET = 'LUAD'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / "human_lung_cancer"
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
STAGE1_CHECKPOINT_ROOT = experiment_path(CONFIG["stage1_checkpoint"])
STAGE2_CHECKPOINT_ROOT = experiment_path(CONFIG["stage2_checkpoint"])
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
TRANSITION_PRIOR = experiment_path(CONFIG["transition_prior_path"]) if "transition_prior_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)



In [3]:
import re
import scanpy as sc
import torch
import numpy as np
import pandas as pd
import sys

import scvi
from pathlib import Path
import importlib
from stvirtual.models import stage1_3d as s1
from stvirtual.models import stage2_3d_transition as s2


In [4]:
data_path = str(DATA_ROOT)
ckpt_path = str(CHECKPOINT_ROOT)
resl_path = str(RUN_ROOT)
lrpr_path = str(LR_PAIRS)
diff_path = str(TRANSITION_PRIOR)


In [5]:
route_ids=['AAH', 'LUAD']
fracs = ['AAH_to_LUAD']
seg_key = "AAH_to_LUAD" 
steps = 100


In [6]:
adata = sc.read_h5ad(SCANVI_ADATA)
adata


AnnData object with n_obs × n_vars = 4280 × 18085
    obs: 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres', 'sample_id', 'condition', 'patient', 'status', 'clone', 'tissue_need', '_sample_name', '_clone_s', '_scvi_batch', '_scvi_labels', 'cx_aligned', 'cy_aligned'
    var: 'gene_ids', 'feature_types'
    uns: '_clone_s_colors', '_scvi_manager_uuid', '_scvi_uuid', 'log1p', 'neighbors', 'pca', 'sample_id_colors', 'status_colors', 'umap'
    obsm: 'X_pca', 'X_scanVI', 'X_umap', 'spatial', 'spatial_aligned'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [7]:
adata.obs['status'].unique()


['AAH', 'LUAD']
Categories (2, object): ['AAH', 'LUAD']

In [8]:
model = None


## Train Stage 1


In [9]:
importlib.reload(s1)

res1 = s1.train_model_multislice(
    model=model,                  
    adata_all=adata,
    slice_key="status",
    route_ids=route_ids,
    save_root=str(STAGE1_CHECKPOINT_ROOT),
    x_key="cx_aligned",
    y_key="cy_aligned",
    latent_key=CONFIG["latent_key"],    
    steps=steps,  
    guide_eps=0.01,
    uot_eps=CONFIG["stage1_loss"]["uot_eps"], uot_tau=CONFIG["stage1_loss"]["uot_tau"], uot_lam_x=CONFIG["stage1_loss"]["uot_lam_x"], uot_lam_f=CONFIG["stage1_loss"]["uot_lam_f"],
    guide_topk=32, guide_temp=0.5, guide_schedule="linear",
    terrain_gap=False,
    cell_type_key='_clone_s',
    lam_context=0.0,
    lam_residual=1.0,
    lam_vsmooth=0.1,  
    lam_uot=CONFIG["stage1_loss"]["lam_uot"],
    lib_layer='counts',              
    latent_dim=10,             
    epochs=training_epochs(100),
    device="cuda:0",
)



[MultiSlice] Train segment: AAH_to_LUAD (n_src=806, n_tgt=3474)
  Global norm route_ids=['AAH', 'LUAD']  latent_dim=10



Train (NeuralODE dopri5):   0%|          | 0/100 [00:00<?, ?it/s]

Train (NeuralODE dopri5):   0%|          | 0/100 [00:00<?, ?it/s, loss=0.0022, uot=0.0015, vs=0.0064]

Train (NeuralODE dopri5):   1%|          | 1/100 [00:00<01:30,  1.10it/s, loss=0.0022, uot=0.0015, vs=0.0064]

Train (NeuralODE dopri5):   1%|          | 1/100 [00:01<01:30,  1.10it/s, loss=0.0020, uot=0.0015, vs=0.0058]

...

Train (NeuralODE dopri5):  98%|█████████▊| 98/100 [00:35<00:00,  2.81it/s, loss=0.0012, uot=0.0006, vs=0.0043]

Train (NeuralODE dopri5):  99%|█████████▉| 99/100 [00:35<00:00,  2.84it/s, loss=0.0012, uot=0.0006, vs=0.0043]

Train (NeuralODE dopri5):  99%|█████████▉| 99/100 [00:35<00:00,  2.84it/s, loss=0.0013, uot=0.0006, vs=0.0054]

Train (NeuralODE dopri5): 100%|██████████| 100/100 [00:36<00:00,  2.81it/s, loss=0.0013, uot=0.0006, vs=0.0054]

Train (NeuralODE dopri5): 100%|██████████| 100/100 [00:36<00:00,  2.77it/s, loss=0.0013, uot=0.0006, vs=0.0054]

In [10]:
saved = {'AAH_to_LUAD': str(CHECKPOINT_ROOT / 'stage1_res' / 'simulation_stage1_AAH_to_LUAD.npz')}


## Create boundaries


In [11]:
from stvirtual.utils import boundary_2d as boundary
from stvirtual.utils.stage1_results import save_res as save_stage1_results
from stvirtual.utils.trajectory import simulation_trace_from_out

boundary_cfg = CONFIG["boundary"]
saved = save_stage1_results(
    res1=res1, adata_all=adata,
    out_dir=str(CHECKPOINT_ROOT / "stage1_res"),
    slice_key="status", ann_key="_clone_s",
    save_prefix="simulation_stage1", steps=steps,
    n_cache=boundary_cfg["n_cache"], unnormalize=False,
)

for stage_key in fracs:
    source_name = stage_key.split("_to_", 1)[0]
    match = re.search(r"\d+", source_name)
    source_number = int(match.group(0)) if match else 10**9
    trace = simulation_trace_from_out(res1[stage_key], steps=steps, n_cache=boundary_cfg["n_cache"], unnormalize=False)
    frames = [value.detach().cpu().numpy().astype(np.float32) for value in trace["x"]]
    stage_dir = RUN_ROOT / "bound" / stage_key
    stage_dir.mkdir(parents=True, exist_ok=True)
    statistics = []
    for frame_index, coordinates in enumerate(frames):
        shell, fraction, outside, expansion = boundary.make_shell_adaptive(
            coordinates, alpha_factor=boundary_cfg["alpha_factor"],
            seed=2026 + source_number + frame_index,
            n_resample=boundary_cfg["n_resample"], target_frac=boundary_cfg["target_frac"],
            expand0=boundary_cfg["expand0"], expand_step=boundary_cfg["expand_step"],
            expand_max=boundary_cfg["expand_max"], fallback_expand=boundary_cfg["fallback_expand"],
        )
        boundary.save_shell_csv(shell, stage_dir / f"bound_z{frame_index:03d}.csv")
        boundary.plot_shell_coverage(coordinates, shell, outside, stage_dir / f"bound_z{frame_index:03d}.png")
        statistics.append({"frame": frame_index, "n_cells": len(coordinates), "fraction_inside": fraction, "n_outside": len(outside), "expansion": expansion})
    pd.DataFrame(statistics).to_csv(stage_dir / "bound_stats.csv", index=False)

print("Stage-1 traces:", saved)
print("Boundary root:", RUN_ROOT / "bound")


Stage-1 traces: {'AAH_to_LUAD': '/home/zhouyj/stVirtual_tutorial/experiments/human_lung_cancer/artifacts/checkpoints/stage1_res/simulation_stage1_AAH_to_LUAD.npz'}
Boundary root: /home/zhouyj/stVirtual_tutorial/experiments/human_lung_cancer/artifacts/results/bound


## Train Stage 2


In [12]:
importlib.reload(s2)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ctx = s2.build_global_ctx(
    adata_path=str(SCANVI_ADATA),
    lr_pairs_path=lrpr_path,
    ckpt_3dslice=str(STAGE1_CHECKPOINT_ROOT / "AAH_to_LUAD" / "checkpoints" / "best.pt"),
    device=device,
    layer_col="_clone_s",
)

stages = []
for sk in fracs:
    src, tgt = sk.split("_to_", 1)
    stages.append(
        s2.StageCfg(
            src=src,
            tgt=tgt,
            out_npz_path=saved[sk],
            bound_dir=f"{resl_path}/bound/{sk}",
            decoder_checkpoint=str(experiment_path(CONFIG["decoder_checkpoint"].format(src=src, tgt=tgt))),
            scanvi_dir=None,
            diff_csv=diff_path,
            model_type="scanvi",
            use_lr=True,
            lr_source="decoder",
            use_latent=True,
            latent_key=CONFIG["latent_key"],
            z_csv_offset=0,
            layer_col="_clone_s",  
        )
    )

outs = s2.run_multi_stages(
    ctx=ctx,
    sample_key = 'status',
    stages=stages,
    best_ckpt_dir=str(STAGE2_CHECKPOINT_ROOT),
    train_kwargs=dict(EPOCHS=training_epochs(100), LR=1e-4),
    rl_xy=CONFIG["stage2_loss"]["rl_xy"],
    rl_z=CONFIG["stage2_loss"]["rl_z"],
)

for o in outs:
    print(o["best_ckpt_path"])


[Grid] H=48, W=68, H×W=3264
[auto cap] 3 {'src': {'q': 0.9, 'quantile_value': 1.0, 'mean': 1.0, 'max': 1.0, 'n_cells_used': 804}, 'tgt': {'q': 0.9, 'quantile_value': 3.0, 'mean': 2.0034801959991455, 'max': 3.0, 'n_cells_used': 1724}, 'cap0': 1, 'capT': 3, 'cap': 3}


[diff] loaded /home/zhouyj/stVirtual_tutorial/experiments/human_lung_cancer/data/diff_map.csv Kmax= 2


Training:   0%|          | 0/100 [00:00<?, ?it/s]

Training:   0%|          | 0/100 [00:09<?, ?it/s, rew=-0.1281, best=-0.1281, tgt=0.0644, occ=0.6294]

Training:   1%|          | 1/100 [00:09<16:20,  9.90s/it, rew=-0.1281, best=-0.1281, tgt=0.0644, occ=0.6294]

...

Training:  99%|█████████▉| 99/100 [16:20<00:09,  9.90s/it, rew=-0.1189, best=-0.1178, tgt=0.0576, occ=0.6045]

Training:  99%|█████████▉| 99/100 [16:29<00:09,  9.90s/it, rew=-0.1229, best=-0.1178, tgt=0.0594, occ=0.6283]

Training: 100%|██████████| 100/100 [16:30<00:00,  9.91s/it, rew=-0.1229, best=-0.1178, tgt=0.0594, occ=0.6283]

Training: 100%|██████████| 100/100 [16:30<00:00,  9.90s/it, rew=-0.1229, best=-0.1178, tgt=0.0594, occ=0.6283]

/home/zhouyj/stVirtual_tutorial/experiments/human_lung_cancer/artifacts/checkpoints/stage2/policy_AAH_to_LUAD.pt


In [13]:
ckpt_map = {}
for o in outs:
    cfg = o["stage"]
    ckpt_map[f"{cfg.src}_to_{cfg.tgt}"] = o["best_ckpt_path"]

ckpt_map


{'AAH_to_LUAD': '/home/zhouyj/stVirtual_tutorial/experiments/human_lung_cancer/artifacts/checkpoints/stage2/policy_AAH_to_LUAD.pt'}

## Run simulation


In [14]:
importlib.reload(s2)
simulations = {}
for cfg in stages:
    seg_key = f"{cfg.src}_to_{cfg.tgt}"
    simulations[seg_key] = s2.simulation_policy_one_stage(
        s2, ctx, cfg, ckpt_map[seg_key],sample_key='status',
        seed=2026, ADVECT_LATENT=True,
        output_dir=Path(resl_path) / 'simulation' / seg_key, output_prefix=seg_key,  
    )
    print(seg_key, "Tp1=", len(simulations[seg_key]["coords"]))


[Grid] H=48, W=68, H×W=3264
[auto cap] 3 {'src': {'q': 0.9, 'quantile_value': 1.0, 'mean': 1.0, 'max': 1.0, 'n_cells_used': 804}, 'tgt': {'q': 0.9, 'quantile_value': 3.0, 'mean': 2.0034801959991455, 'max': 3.0, 'n_cells_used': 1724}, 'cap0': 1, 'capT': 3, 'cap': 3}


[diff] loaded /home/zhouyj/stVirtual_tutorial/experiments/human_lung_cancer/data/diff_map.csv Kmax= 2


AAH_to_LUAD Tp1= 101


In [15]:
r0 = simulations['AAH_to_LUAD']
print(r0.keys())


dict_keys(['coords', 'layers', 'latent', 'lr', 'anchor', 'is_birth', 'born_step', 'uid', 'parent_uid', 'event', 'is_diff', 'diff_alpha', 'diff_alpha_post', 'diff_tgt_layer', 'diff_enter_step', 'commit_step', 't', 'T', 'n_layers', 'output_paths'])


In [16]:
import numpy as np
import pandas as pd

fi = 1           # 

layers = np.asarray(r0["layers"][fi]).astype(int)
is_diff = np.asarray(r0["is_diff"][fi]).astype(bool)
diff_tgt = np.asarray(r0["diff_tgt_layer"][fi]).astype(int)
commit = np.asarray(r0["commit_step"][fi]).astype(int)

#  diff :is_diff=True, tgt>=0,  commit
m = is_diff & (diff_tgt >= 0) & (commit < 0)

df = pd.DataFrame({
    "src_layer_internal": layers[m],
    "tgt_layer_internal": diff_tgt[m],
})

print("n_active_diff =", len(df))
print(df.value_counts().sort_index())


n_active_diff = 48
src_layer_internal  tgt_layer_internal
4                   1                     19
                    2                     18
5                   1                      5
                    2                      6
Name: count, dtype: int64


In [17]:
ro = simulations[seg_key]

for t in range(len(ro["coords"])):
    N  = int(np.asarray(ro["coords"][t]).shape[0])
    nb = int(np.asarray(ro["is_birth"][t]).sum())  
    print(f"{seg_key} | t={t:02d} | N={N} | new_birth={nb}")


AAH_to_LUAD | t=00 | N=806 | new_birth=0
AAH_to_LUAD | t=01 | N=833 | new_birth=138
AAH_to_LUAD | t=02 | N=859 | new_birth=134
AAH_to_LUAD | t=03 | N=886 | new_birth=149
AAH_to_LUAD | t=04 | N=913 | new_birth=138
AAH_to_LUAD | t=05 | N=940 | new_birth=144
AAH_to_LUAD | t=06 | N=966 | new_birth=144
AAH_to_LUAD | t=07 | N=993 | new_birth=136
AAH_to_LUAD | t=08 | N=1020 | new_birth=169
AAH_to_LUAD | t=09 | N=1047 | new_birth=171
AAH_to_LUAD | t=10 | N=1073 | new_birth=177
AAH_to_LUAD | t=11 | N=1100 | new_birth=182
AAH_to_LUAD | t=12 | N=1127 | new_birth=178
AAH_to_LUAD | t=13 | N=1154 | new_birth=179
AAH_to_LUAD | t=14 | N=1180 | new_birth=187
AAH_to_LUAD | t=15 | N=1207 | new_birth=190
AAH_to_LUAD | t=16 | N=1234 | new_birth=190
AAH_to_LUAD | t=17 | N=1261 | new_birth=203
AAH_to_LUAD | t=18 | N=1287 | new_birth=204
AAH_to_LUAD | t=19 | N=1314 | new_birth=203
AAH_to_LUAD | t=20 | N=1341 | new_birth=203
AAH_to_LUAD | t=21 | N=1368 | new_birth=212
AAH_to_LUAD | t=22 | N=1394 | new_birth=21

In [18]:
ro.keys()


dict_keys(['coords', 'layers', 'latent', 'lr', 'anchor', 'is_birth', 'born_step', 'uid', 'parent_uid', 'event', 'is_diff', 'diff_alpha', 'diff_alpha_post', 'diff_tgt_layer', 'diff_enter_step', 'commit_step', 't', 'T', 'n_layers', 'output_paths'])

In [19]:
TRANSITION_OBS_FIELDS = (
    "uid",
    "parent_uid",
    "diff_alpha",
    "src_layer_id",
    "src_layer",
    "tgt_layer_id",
    "tgt_layer",
)

def persist_transition_simulation_obs(simulation_result, celltype_names):
    output_paths = simulation_result.get("output_paths")
    required = ("uid", "parent_uid", "layers", "is_diff", "diff_alpha", "diff_tgt_layer", "commit_step")
    missing = [field for field in required if not isinstance(simulation_result.get(field), list)]
    if missing:
        raise KeyError(f"simulation_result is missing transition fields: {missing}")
    if not isinstance(output_paths, list):
        raise KeyError("simulation_result has no output_paths; pass output_dir to simulation_policy_one_stage")
    frame_count = len(simulation_result["coords"])
    if len(output_paths) != frame_count:
        raise ValueError("output_paths and simulation frames have different lengths")

    names = [str(name) for name in celltype_names]
    source_by_uid = {}
    target_by_uid = {}
    for frame_index in range(frame_count):
        uids = np.asarray(simulation_result["uid"][frame_index], dtype=np.int64)
        current_layers = np.asarray(simulation_result["layers"][frame_index], dtype=np.int64)
        target_layers = np.asarray(simulation_result["diff_tgt_layer"][frame_index], dtype=np.int64)
        for uid, current_layer, target_layer in zip(uids, current_layers, target_layers):
            source_by_uid.setdefault(int(uid), int(current_layer))
            if int(target_layer) >= 0:
                target_by_uid[int(uid)] = int(target_layer)

    for frame_index, output_path in enumerate(output_paths):
        frame_adata = sc.read_h5ad(output_path)
        uids = np.asarray(simulation_result["uid"][frame_index], dtype=np.int64)
        parent_uids = np.asarray(simulation_result["parent_uid"][frame_index], dtype=np.int64)
        diff_alpha = np.asarray(simulation_result["diff_alpha"][frame_index], dtype=np.float32)
        is_diff = np.asarray(simulation_result["is_diff"][frame_index], dtype=bool)
        commit_step = np.asarray(simulation_result["commit_step"][frame_index], dtype=np.int32)
        has_transition = is_diff | (commit_step >= 0)

        src_ids = np.full(frame_adata.n_obs, -1, dtype=np.int64)
        tgt_ids = np.full(frame_adata.n_obs, -1, dtype=np.int64)
        for index, uid in enumerate(uids):
            if has_transition[index]:
                src_ids[index] = source_by_uid[int(uid)]
                tgt_ids[index] = target_by_uid.get(int(uid), -1)

        src_names = np.array([names[index] if 0 <= index < len(names) else "" for index in src_ids], dtype=str)
        tgt_names = np.array([names[index] if 0 <= index < len(names) else "" for index in tgt_ids], dtype=str)
        frame_adata.obs["uid"] = uids
        frame_adata.obs_names = uids.astype(str)
        frame_adata.obs["parent_uid"] = parent_uids
        frame_adata.obs["diff_alpha"] = diff_alpha
        frame_adata.obs["src_layer_id"] = src_ids
        frame_adata.obs["src_layer"] = src_names
        frame_adata.obs["tgt_layer_id"] = tgt_ids
        frame_adata.obs["tgt_layer"] = tgt_names
        frame_adata.write_h5ad(output_path, compression="gzip")

    return {"frames_updated": frame_count, "obs_fields": list(TRANSITION_OBS_FIELDS)}

simulation_obs_reports = {
    route: persist_transition_simulation_obs(route_simulation, ctx.layers_list)
    for route, route_simulation in simulations.items()
}
simulation_obs_reports


{'AAH_to_LUAD': {'frames_updated': 101,
  'obs_fields': ['uid',
   'parent_uid',
   'diff_alpha',
   'src_layer_id',
   'src_layer',
   'tgt_layer_id',
   'tgt_layer']}}

In [20]:
from _shared.runtime import save_run_gallery

overview_path = save_run_gallery(EXPERIMENT_DIR)


Saved 22 panels: /home/zhouyj/stVirtual_tutorial/experiments/human_lung_cancer/artifacts/results/overview/AAH_to_LUAD.png
